# ChatPromptTemplate
> ChatPromptTemplate -> 역할 기반 대화 구조

### 예제
## ChatPromptTemplate

In [132]:
template = """
이전 대화:
{history}

사용자 요청:
{user_input}

{format_instructions}
"""

In [133]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage # 모델에게 역할과 규칙을 주는 시스템 메시지
from langchain_core.prompts import HumanMessagePromptTemplate # 사용자 입력이 들어갈 human 메시지 템플릿

# 시스템과 휴먼 역할을 정의한 채팅 프롬프트 템플릿 생성
chat_prompt = ChatPromptTemplate(
    # 채팅 이력 데이터들
    messages=[
        # system 역할
        SystemMessage(
            # AI 모델이 부여된 역할
            content=(
                "당신은 친근하고 도움이 되는 AI 어이스턴트 입니다. 항상 한국어로 답변해주세요"
            )
        ),
        # human 역할
        # 사람이 사용할 템플릿
        HumanMessagePromptTemplate.from_template(
            template=template
        )
    ]
)

In [134]:
chat_prompt.input_variables

['format_instructions', 'history', 'user_input']

### Groq API Key

In [112]:
from dotenv import load_dotenv

# dotenv 파일에서 환경변수 로드
load_dotenv()

True

### LLM

In [113]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model = "openai/gpt-oss-120b",  # 모델명
    temperature = 0.1,              # 낮을수록 보통 더 일관적이고 덜 랜덤한 응답 경향
    model_kwargs = {
        "top_p":0.9,                # 확률 질량 누적 0.9 안에서 다음 토큰 후보를 샘플링
        # 그럴듯한 단어들만 골라서 그 안에서 랜덤 선택
        "frequency_penalty":0.7,    # 반복적인 단어/표현이 다시 나오는 걸 줄이는 성향
        "presence_penalty":0.6,     # 새로운 아이디어 유도
    },
    max_tokens=2000
)

# LangChain Expression Language

> 사용자 입력 -> 프롬프트 구성 -> LLM 호출 -> 결과 반환

In [114]:
# chain 생성
chain = chat_prompt | model
# 프롬프트 | 모델 : 이런 형태

In [115]:
chat_prompt.input_variables

['user_input']

In [125]:
chat_history = [
    {"role": "user", "content": "안녕"},
    {"role": "assistant", "content": "안녕하세요! 무엇을 도와드릴까요?"},
    {"role": "user", "content": "내 이름은 홍길동이야"}
]

In [126]:
from langchain_core.messages import HumanMessage, AIMessage


def chat(model, chat_prompt, user_input, chat_history):
    chain = chat_prompt | model

    response = chain.invoke({
        "chat_history": chat_history,
        "user_input": user_input
    })

    # 실제 대화 이력 저장
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=response.content))

    return response.content


while True:
    user_input = input("사용자: ")
    
    if user_input == "끝":
        print("대화를 종료합니다.")
        break

    answer = chat(model, chat_prompt, user_input, chat_history)
    print("AI:", answer)

AI: 안녕하세요! 😊 무엇을 도와드릴까요? 언제든 편하게 말씀해 주세요.
AI: 아직 제게는 당신의 이름이 알려져 있지 않아요! 혹시 알려주시면, 앞으로 더 친근하게 대화할 수 있을 거예요. 😊
AI: 안녕하세요! 무엇을 도와드릴까요? 궁금한 점이나 도움이 필요한 부분이 있으면 편하게 말씀해주세요. 😊
대화를 종료합니다.


In [119]:
# chain 호출
response = chain.invoke({
    "user_input":"중력에 대해 설명해줘."
})

In [120]:
# 모델 응답 객체 출력
response

AIMessage(content='## 중력(重力)이란?\n\n**중력**은 모든 물체가 서로 끌어당기는 힘을 말합니다. 우리 주변에서 가장 쉽게 느낄 수 있는 힘 중 하나이며, 지구가 우리를 땅에 붙잡아 두는 이유이기도 합니다. \n\n---\n\n## 1️⃣ 뉴턴의 만유인력 법칙\n\n### 핵심 아이디어\n- **아이작 뉴턴(Isaac Newton)**이 1687년에 제시한 법칙으로, “두 물체 사이에는 질량에 비례하고 거리의 제곱에 반비례하는 인력이 작용한다”는 내용입니다.\n\n### 수식\n\\[\nF = G \\frac{m_1 m_2}{r^2}\n\\]\n\n- **F** : 두 물체 사이에 작용하는 중력 (단위: 뉴턴, N)  \n- **G** : 만유인력 상수 \\(\\approx 6.674 \\times 10^{-11}\\, \\text{Nm}^2/\\text{kg}^2\\)  \n- **m₁, m₂** : 각각의 물체 질량 (kg)  \n- **r** : 두 물체 중심 사이 거리 (m)\n\n### 일상 속 예시\n- **지구와 사과**: 사과가 떨어지는 이유는 지구와 사과 사이에 작용하는 중력 때문입니다.  \n- **달과 지구**: 달이 지구 주위를 도는 것도 중력 덕분이에요.\n\n---\n\n## 2️⃣ 아인슈타인의 일반 상대성 이론\n\n### 왜 필요했나요?\n- 뉴턴의 법칙은 일상적인 속도와 중력에서는 아주 정확하지만, **극도로 강한 중력**(예: 블랙홀)이나 **빛과 같은 빠른 물체**에 대해서는 한계가 있었습니다.\n\n### 핵심 개념\n- **시공간(시간 + 공간)**이 물질에 의해 **구부러진다**는 것이 핵심입니다.  \n- 물체는 이 휘어진 시공간을 따라 “자연스럽게” 움직이며, 이것이 **중력**으로 관측됩니다.\n\n### 비유\n- **고무 시트 위에 무거운 공을 올려놓는 것**을 생각해 보세요. 공이 시트를 눌러서 움푹 파이고, 그 주변에 작은 공을 올리면 작은 공이 큰 공 쪽으로 굴러갑니다. 여기서 시트가

In [121]:
# 대답에 대한 내용부분만 출력
print(response.content)

## 중력(重力)이란?

**중력**은 모든 물체가 서로 끌어당기는 힘을 말합니다. 우리 주변에서 가장 쉽게 느낄 수 있는 힘 중 하나이며, 지구가 우리를 땅에 붙잡아 두는 이유이기도 합니다. 

---

## 1️⃣ 뉴턴의 만유인력 법칙

### 핵심 아이디어
- **아이작 뉴턴(Isaac Newton)**이 1687년에 제시한 법칙으로, “두 물체 사이에는 질량에 비례하고 거리의 제곱에 반비례하는 인력이 작용한다”는 내용입니다.

### 수식
\[
F = G \frac{m_1 m_2}{r^2}
\]

- **F** : 두 물체 사이에 작용하는 중력 (단위: 뉴턴, N)  
- **G** : 만유인력 상수 \(\approx 6.674 \times 10^{-11}\, \text{Nm}^2/\text{kg}^2\)  
- **m₁, m₂** : 각각의 물체 질량 (kg)  
- **r** : 두 물체 중심 사이 거리 (m)

### 일상 속 예시
- **지구와 사과**: 사과가 떨어지는 이유는 지구와 사과 사이에 작용하는 중력 때문입니다.  
- **달과 지구**: 달이 지구 주위를 도는 것도 중력 덕분이에요.

---

## 2️⃣ 아인슈타인의 일반 상대성 이론

### 왜 필요했나요?
- 뉴턴의 법칙은 일상적인 속도와 중력에서는 아주 정확하지만, **극도로 강한 중력**(예: 블랙홀)이나 **빛과 같은 빠른 물체**에 대해서는 한계가 있었습니다.

### 핵심 개념
- **시공간(시간 + 공간)**이 물질에 의해 **구부러진다**는 것이 핵심입니다.  
- 물체는 이 휘어진 시공간을 따라 “자연스럽게” 움직이며, 이것이 **중력**으로 관측됩니다.

### 비유
- **고무 시트 위에 무거운 공을 올려놓는 것**을 생각해 보세요. 공이 시트를 눌러서 움푹 파이고, 그 주변에 작은 공을 올리면 작은 공이 큰 공 쪽으로 굴러갑니다. 여기서 시트가 **시공간**, 큰 공이 **질량**이며, 작은 공이 **다른 물체**입니다.

### 중요한 결과
- **빛도 휘어